# 02 — Preprocessing Pipeline (80/20 Design)

**Inputs:** same `eicu_features.csv`, `mimic_features.csv`

**Outputs:** new files saved alongside originals — nothing overwritten:

| File | Description |
|---|---|
| `X_train_80.csv` | 2,016 eICU patients (80%), scaler fit on these |
| `y_train_80.csv` | labels |
| `X_val_80.csv` | 504 eICU patients (20%), transformed with new scaler |
| `y_val_80.csv` | labels |
| `X_mimic_80.csv` | 136 MIMIC patients, transformed with new scaler |
| `X_train_smote_80.csv` | SMOTE-resampled training set |
| `preprocessing_objects_80.pkl` | scaler/imputer fit on 80% train |

Original files (`X_train.csv`, `X_val.csv`, `X_test.csv`, etc.) are **not touched**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install -q imbalanced-learn
import pandas as pd
import numpy as np
import warnings, os, pickle
warnings.filterwarnings('ignore')

from sklearn.impute          import SimpleImputer
from sklearn.preprocessing   import StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling  import SMOTE
from collections             import Counter
from scipy.stats             import ks_2samp

SEED = 42
np.random.seed(SEED)

BASE       = '/content/drive/MyDrive/AI in Medicine/data/output_data'
EICU_PATH  = f'{BASE}/eicu_train/eicu_features.csv'
MIMIC_PATH = f'{BASE}/mimic_val/mimic_features.csv'
OUT_DIR    = f'{BASE}/preprocessed'   # same folder — new file names prevent overwrites
os.makedirs(OUT_DIR, exist_ok=True)

print('Libraries loaded.')

Libraries loaded.


## Steps 1–4: Column Harmonization, Leakage Removal, Missingness Drop, Redundant Drop

Identical to notebook 02.

In [ ]:
eicu  = pd.read_csv(EICU_PATH)
mimic = pd.read_csv(MIMIC_PATH)
print(f'eICU : {eicu.shape}   mortality={eicu["mortality"].mean():.3f}')
print(f'MIMIC: {mimic.shape}  mortality={mimic["mortality"].mean():.3f}')

# ── Step 1: Column harmonization — eICU → MIMIC naming convention ─────────────
VITAL_RENAME = {}
for stat in ['min', 'max', 'mean']:
    VITAL_RENAME[f'heartrate_{stat}']    = f'hr_{stat}'
    VITAL_RENAME[f'respiration_{stat}']  = f'rr_{stat}'
    VITAL_RENAME[f'sao2_{stat}']         = f'spo2_{stat}'
    VITAL_RENAME[f'ssystolic_{stat}']    = f'sbp_{stat}'
    VITAL_RENAME[f'sdiastolic_{stat}']   = f'dbp_{stat}'
    VITAL_RENAME[f'systemicmean_{stat}'] = f'map_{stat}'
eicu = eicu.rename(columns=VITAL_RENAME)

# ── Step 2: Align columns ─────────────────────────────────────────────────────
shared    = sorted(set(eicu.columns) & set(mimic.columns))
keep_cols = [c for c in shared if c not in ['hadm_id']]
eicu  = eicu[keep_cols].copy()
mimic = mimic[keep_cols].copy()

# ── Step 3: Drop leaky / ID columns ──────────────────────────────────────────
DROP_COLS = ['stay_id', 'dataset', 'icu_los_days']
eicu  = eicu.drop(columns=[c for c in DROP_COLS if c in eicu.columns])
mimic = mimic.drop(columns=[c for c in DROP_COLS if c in mimic.columns])

# ── Step 4a: Drop high-missingness (>70%) — protect BP/INR ───────────────────
miss_eicu = eicu.isnull().mean()
PROTECT   = ['lactate_max','lactate_min','lactate_mean',
             'bun_max','bun_min','bun_mean',
             'sbp_max','sbp_min','sbp_mean',
             'dbp_max','dbp_min','dbp_mean',
             'map_max','map_min','map_mean',
             'inr_max','inr_min','inr_mean']
cols_to_drop = [c for c in miss_eicu[miss_eicu > 0.70].index if c not in PROTECT]
eicu  = eicu.drop(columns=cols_to_drop)
mimic = mimic.drop(columns=[c for c in cols_to_drop if c in mimic.columns])

# ── Step 4b: Drop redundant _mean lab columns ─────────────────────────────────
LAB_BASES    = ['albumin','alt','bicarbonate','bun','creatinine','glucose',
                'hematocrit','hemoglobin','inr','lactate','platelets',
                'potassium','sodium','wbc']
LAB_MEAN_COLS = [f'{b}_mean' for b in LAB_BASES if f'{b}_mean' in eicu.columns]
eicu  = eicu.drop(columns=LAB_MEAN_COLS)
mimic = mimic.drop(columns=[c for c in LAB_MEAN_COLS if c in mimic.columns])

print(f'\nAfter steps 1–4:  eICU {eicu.shape}  |  MIMIC {mimic.shape}')

eICU : (2520, 73)   mortality=0.050
MIMIC: (136, 74)  mortality=0.338

After steps 1–4:  eICU (2520, 53)  |  MIMIC (136, 53)


## Step 5: 80/20 Split ← KEY CHANGE

Original notebook 02 used 60/20/20. Here we use **80/20** — no internal test set. MIMIC is the sole test set.

In [ ]:
TARGET  = 'mortality'
X_eicu  = eicu.drop(columns=[TARGET])
y_eicu  = eicu[TARGET].astype(int)
X_mimic = mimic.drop(columns=[TARGET])
y_mimic = mimic[TARGET].astype(int)

# ── 80/20 split — single split, no test set ───────────────────────────────────
X_train, X_val, y_train, y_val = train_test_split(
    X_eicu, y_eicu,
    test_size=0.20,
    stratify=y_eicu,
    random_state=SEED
)

print('80/20 Split:')
print(f'  Train : {X_train.shape}  mortality={y_train.mean():.3f}')
print(f'  Val   : {X_val.shape}    mortality={y_val.mean():.3f}')
print(f'  MIMIC : {X_mimic.shape}  mortality={y_mimic.mean():.3f}')
print(f'\nClass distribution (train): {Counter(y_train)}')

80/20 Split:
  Train : (2016, 52)  mortality=0.050
  Val   : (504, 52)    mortality=0.050
  MIMIC : (136, 52)  mortality=0.338

Class distribution (train): Counter({0: 1915, 1: 101})


## Steps 6–10: Outlier Capping, Missingness Flags, Imputation, OHE, Scaling

All fitting on 80% train only — val and MIMIC are only transformed.

In [ ]:
numeric_cols = X_train.select_dtypes(include=np.number).columns.tolist()

# ── Step 6: Outlier capping (fit on train) ────────────────────────────────────
clip_bounds = {}
for col in numeric_cols:
    lo = X_train[col].quantile(0.01)
    hi = X_train[col].quantile(0.99)
    clip_bounds[col] = (lo, hi)
    for df in [X_train, X_val, X_mimic]:
        df[col] = df[col].clip(lo, hi)

print(f'Step 6: Winsorisation applied to {len(numeric_cols)} numeric features.')

# ── Step 7: Missingness flags (before imputation) ────────────────────────────
FLAG_COLS = []
for base in ['albumin', 'lactate', 'alt', 'sbp', 'dbp', 'map', 'inr']:
    related = [c for c in numeric_cols if c.startswith(base + '_') and 'was_missing' not in c]
    if not related:
        continue
    flag = f'{base}_was_missing'
    for df in [X_train, X_val, X_mimic]:
        df[flag] = df[related].isnull().all(axis=1).astype(int)
    FLAG_COLS.append(flag)
print(f'Step 7: {len(FLAG_COLS)} missingness flags added: {FLAG_COLS}')

# ── Step 8: Imputation (fit on train) ────────────────────────────────────────
numeric_to_impute = [c for c in X_train.select_dtypes(include=np.number).columns if c not in FLAG_COLS]
num_imputer = SimpleImputer(strategy='median')
num_imputer.fit(X_train[numeric_to_impute])
for df in [X_train, X_val, X_mimic]:
    df[numeric_to_impute] = num_imputer.transform(df[numeric_to_impute])

cat_cols = X_train.select_dtypes(include='object').columns.tolist()
if cat_cols:
    cat_imputer = SimpleImputer(strategy='most_frequent')
    cat_imputer.fit(X_train[cat_cols])
    for df in [X_train, X_val, X_mimic]:
        df[cat_cols] = cat_imputer.transform(df[cat_cols])
print(f'Step 8: Imputation done. NaN remaining: {X_train.isnull().sum().sum()}')

# ── Step 9: One-hot encoding ──────────────────────────────────────────────────
CAT_ENCODE = [c for c in ['careunit','admit_source','ethnicity_grp','icd9_chapter'] if c in X_train.columns]
X_train = pd.get_dummies(X_train, columns=CAT_ENCODE, drop_first=True, dtype=int)
train_cols = X_train.columns.tolist()
X_val   = pd.get_dummies(X_val,   columns=CAT_ENCODE, drop_first=True, dtype=int).reindex(columns=train_cols, fill_value=0)
X_mimic = pd.get_dummies(X_mimic, columns=CAT_ENCODE, drop_first=True, dtype=int).reindex(columns=train_cols, fill_value=0)
print(f'Step 9: OHE done. Features: {X_train.shape[1]}')

# ── Step 10: Scaling (fit on train, continuous only) ─────────────────────────
binary_names    = set(['gender_binary'] + FLAG_COLS)
ohe_cols        = [c for c in X_train.select_dtypes(include=np.number).columns
                   if any(c.startswith(b + '_') for b in CAT_ENCODE)]
continuous_cols = [c for c in X_train.select_dtypes(include=np.number).columns
                   if c not in set(ohe_cols) | binary_names]

scaler = StandardScaler()
scaler.fit(X_train[continuous_cols])

def scale_df(df, cont_cols, fitted_scaler):
    df_out = df.copy()
    df_out[cont_cols] = fitted_scaler.transform(df[cont_cols])
    return df_out

X_train_scaled = scale_df(X_train, continuous_cols, scaler)
X_val_scaled   = scale_df(X_val,   continuous_cols, scaler)
X_mimic_scaled = scale_df(X_mimic, continuous_cols, scaler)
print(f'Step 10: Scaling applied to {len(continuous_cols)} continuous features.')

Step 6: Winsorisation applied to 48 numeric features.
Step 7: 7 missingness flags added: ['albumin_was_missing', 'lactate_was_missing', 'alt_was_missing', 'sbp_was_missing', 'dbp_was_missing', 'map_was_missing', 'inr_was_missing']
Step 8: Imputation done. NaN remaining: 0
Step 9: OHE done. Features: 87
Step 10: Scaling applied to 47 continuous features.


## Step 11: SMOTE and Step 12: Feature Stability Filtering

Same logic as notebook 02 — SMOTE applied to 80% train. Stability filtering computes KS/Cohen's d/corr delta between 80% train and MIMIC.

In [ ]:
# ── Step 11: SMOTE on 80% train ───────────────────────────────────────────────
k = min(5, Counter(y_train)[1] - 1)
smote = SMOTE(random_state=SEED, k_neighbors=k)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)
print(f'Step 11: SMOTE — before {Counter(y_train)} → after {Counter(y_train_smote)}')

# ── Step 12: Feature stability filtering ─────────────────────────────────────
records = []
for col in X_train_scaled.columns:
    a = X_train_scaled[col].values
    b = X_mimic_scaled[col].values
    ks_stat, _ = ks_2samp(a, b)
    pooled_std  = np.sqrt((a.std()**2 + b.std()**2) / 2 + 1e-8)
    cohen_d     = abs(a.mean() - b.mean()) / pooled_std
    corr_train  = abs(np.corrcoef(X_train_scaled[col], y_train)[0, 1])
    corr_mimic  = abs(np.corrcoef(X_mimic_scaled[col], y_mimic)[0, 1])
    corr_delta  = corr_train - corr_mimic
    records.append({'feature': col, 'ks_stat': ks_stat, 'cohen_d': cohen_d, 'corr_delta': corr_delta})

instability_df = pd.DataFrame(records)
for c in ['ks_stat', 'cohen_d', 'corr_delta']:
    instability_df[f'rank_{c}'] = instability_df[c].rank(ascending=True)
instability_df['instability_score'] = instability_df[['rank_ks_stat','rank_cohen_d','rank_corr_delta']].sum(axis=1)
instability_df = instability_df.sort_values('instability_score', ascending=False).reset_index(drop=True)

# Drop features that fail ≥2 of 3 thresholds
unstable_mask = (
    (instability_df['ks_stat']    > 0.30).astype(int) +
    (instability_df['cohen_d']    > 0.70).astype(int) +
    (instability_df['corr_delta'] > 0.50).astype(int)
) >= 2

unstable_features = instability_df.loc[unstable_mask, 'feature'].tolist()
stable_features   = [f for f in X_train_scaled.columns if f not in unstable_features]

print(f'Step 12: Dropped {len(unstable_features)} unstable features: {unstable_features}')
print(f'         Kept {len(stable_features)} stable features.')

X_train_scaled = X_train_scaled[stable_features]
X_val_scaled   = X_val_scaled[stable_features]
X_mimic_scaled = X_mimic_scaled[stable_features]
X_train_smote  = pd.DataFrame(X_train_smote, columns=instability_df['feature'].tolist())[stable_features]

print(f'\nFinal shapes:')
print(f'  X_train_scaled : {X_train_scaled.shape}  mortality={y_train.mean():.3f}')
print(f'  X_val_scaled   : {X_val_scaled.shape}    mortality={y_val.mean():.3f}')
print(f'  X_mimic_scaled : {X_mimic_scaled.shape}  mortality={y_mimic.mean():.3f}')
print(f'  SMOTE train    : {X_train_smote.shape}')

Step 11: SMOTE — before Counter({0: 1915, 1: 101}) → after Counter({0: 1915, 1: 1915})
Step 12: Dropped 6 unstable features: ['dbp_was_missing', 'sbp_was_missing', 'map_was_missing', 'inr_was_missing', 'sbp_max', 'icd9_chapter_Circulatory']
         Kept 81 stable features.

Final shapes:
  X_train_scaled : (2016, 81)  mortality=0.050
  X_val_scaled   : (504, 81)    mortality=0.050
  X_mimic_scaled : (136, 81)  mortality=0.338
  SMOTE train    : (3830, 81)


## Save — New File Names, Nothing Overwritten

In [ ]:
# ── Save all outputs with _80 suffix — originals untouched ────────────────────
X_train_scaled.to_csv(f'{OUT_DIR}/X_train_80.csv', index=False)
y_train.to_csv(       f'{OUT_DIR}/y_train_80.csv', index=False)

X_val_scaled.to_csv(  f'{OUT_DIR}/X_val_80.csv',   index=False)
y_val.to_csv(         f'{OUT_DIR}/y_val_80.csv',    index=False)

X_mimic_scaled.to_csv(f'{OUT_DIR}/X_mimic_80.csv', index=False)
y_mimic.to_csv(       f'{OUT_DIR}/y_mimic_80.csv',  index=False)

X_train_smote.to_csv( f'{OUT_DIR}/X_train_smote_80.csv', index=False)
pd.Series(y_train_smote, name='mortality').to_csv(
    f'{OUT_DIR}/y_train_smote_80.csv', index=False)

preprocessing_80 = {
    'num_imputer'    : num_imputer,
    'scaler'         : scaler,
    'clip_bounds'    : clip_bounds,
    'continuous_cols': continuous_cols,
    'train_cols'     : train_cols,
    'stable_features': stable_features,
    'flag_cols'      : FLAG_COLS,
    'cat_encode'     : CAT_ENCODE,
}
with open(f'{OUT_DIR}/preprocessing_objects_80.pkl', 'wb') as f:
    pickle.dump(preprocessing_80, f)

print('Saved (nothing overwritten):')
for fname in ['X_train_80.csv','y_train_80.csv','X_val_80.csv','y_val_80.csv',
              'X_mimic_80.csv','y_mimic_80.csv',
              'X_train_smote_80.csv','y_train_smote_80.csv',
              'preprocessing_objects_80.pkl']:
    print(f'  {OUT_DIR}/{fname}')

Saved (nothing overwritten):
  /content/drive/MyDrive/AI in Medicine/data/output_data/preprocessed/X_train_80.csv
  /content/drive/MyDrive/AI in Medicine/data/output_data/preprocessed/y_train_80.csv
  /content/drive/MyDrive/AI in Medicine/data/output_data/preprocessed/X_val_80.csv
  /content/drive/MyDrive/AI in Medicine/data/output_data/preprocessed/y_val_80.csv
  /content/drive/MyDrive/AI in Medicine/data/output_data/preprocessed/X_mimic_80.csv
  /content/drive/MyDrive/AI in Medicine/data/output_data/preprocessed/y_mimic_80.csv
  /content/drive/MyDrive/AI in Medicine/data/output_data/preprocessed/X_train_smote_80.csv
  /content/drive/MyDrive/AI in Medicine/data/output_data/preprocessed/y_train_smote_80.csv
  /content/drive/MyDrive/AI in Medicine/data/output_data/preprocessed/preprocessing_objects_80.pkl
